In [1]:
import streamlit as st
import pandas as pd
from typing import List, Dict, Optional, Any
from openpyxl import load_workbook
import plotly.express as px


In [3]:


def clean_group(df: pd.DataFrame,bundesländer_cols:list) -> pd.DataFrame:
   # Replace '/' with '0' and convert 'Insgesamt' column to integer
   df = df.replace('/', '0')
   df['Insgesamt'] = df['Insgesamt'].astype(int)
   
   # Calculate the total and percentage
   total = df['Insgesamt'].iloc[0]
   df['percent'] = (df['Insgesamt'] / total) * 100
   
   # Drop unnecessary columns and the first row
   df = df.drop(columns=bundesländer_cols)
   df = df.drop(index=0)
   
   return df

#@st.cache_data()
def read_data(file_path: str) -> (Dict[str, pd.DataFrame], List[str]):

    def load_sheets(file_path: str, sheet_names: List[str], header: int = 3) -> Dict[str, pd.DataFrame]:
        return {name: pd.read_excel(file_path, sheet_name=name, header=header) for name in sheet_names}
    
    sheet_names = pd.ExcelFile(file_path).sheet_names
    sheet_data = load_sheets(file_path, sheet_names[3:7])
    return sheet_data, sheet_names

def build_data_frames():
    file_path = '/Users/leonardhaas/code/streamlit/data/raw_data/Zensus22_Sonderauswertung_Haas.xlsx'

    sheet_data,sheet_names = read_data(file_path)   
    haupt_gruppen_1 = sheet_data[sheet_names[3]]
    berufs_gruppen_2 = sheet_data[sheet_names[4]]
    berufs_unter_gruppen_3 = sheet_data[sheet_names[5]]
    berufs_gattungen_4 = sheet_data[sheet_names[6]]


    bundesländer_cols =['Baden-Württemberg', 'Bayern',
        'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen',
        'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen',
        'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt',
        'Schleswig-Holstein', 'Thüringen']

    haupt_gruppen_1 = clean_group(haupt_gruppen_1, bundesländer_cols)
    berufs_gruppen_2 = clean_group(berufs_gruppen_2, bundesländer_cols)
    berufs_unter_gruppen_3 = clean_group(berufs_unter_gruppen_3, bundesländer_cols)
    berufs_gattungen_4 = clean_group(berufs_gattungen_4, bundesländer_cols)

    dataframes = {
        'Hauptgruppe (1-Str.)': haupt_gruppen_1,
        'Berufsgruppe (2-Str.)': berufs_gruppen_2,
        'Berufsuntergruppen (3-St.)': berufs_unter_gruppen_3,
        'Berufsgattung (4-St.)': berufs_gattungen_4
    }
    return dataframes

In [4]:
data_frames = build_data_frames()

In [20]:
data_level_4 = data_frames['Berufsgattung (4-St.)']
data_level_4['merg_id']=data_level_4['ISCO-Code'].map(lambda x: str(x)[:2])

In [23]:
data_level_1 = data_frames['Hauptgruppe (1-Str.)']

In [24]:
data_level_1.merge(data_level_4,left_on='ISCO-Code',right_on='merg_id')

,ISCO-Code_x,Bezeichnung_x,Insgesamt_x,percent_x,ISCO-Code_y,Bezeichnung_y,Insgesamt_y,percent_y,merg_id


ISCO-Code       object
Bezeichnung     object
Insgesamt        int64
percent        float64
merg_id         object
dtype: object

## plot restaurantsbills like chart
https://plotly.com/python/horizontal-bar-charts/